In [ ]:
print("Hello to you!")

# Data import
In this code block you import the data, you can play the game yourself a few times or use our example data.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

df = pd.read_json('game_events.jsonl', lines=True)

df

## Playing around with Pandas
- Find all events where the exit was found
- Sort all exit events by score descending
- Show only the columns: timestamp, score and game_id

In [ ]:
df.loc[df['type']=='ExitFoundEvent']

In [ ]:
df.loc[df['type']=='ExitFoundEvent'].sort_values('score', ascending=False)

In [ ]:
df.loc[df['type']=='ExitFoundEvent', ['timestamp', 'score', 'game_id']].sort_values('score', ascending=False)

# Data Visualisations

In this part, you play around with different plotting mechanisms. You have to work on your pandas. Through the links in the assignments you have the documentation.

## 1. Timeline of the agent’s score

Create a lineplot using the timestamp on the xaxis and the score on the yaxis, create a line per game_id

<b>Solution 1</b>

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# ensure timestamp is datetime
df["timestamp"] = pd.to_datetime(df["timestamp"])

plt.figure()

for game_id, group in df.groupby("game_id"):
    group = group.sort_values("timestamp")
    plt.plot(group["timestamp"], group["score"], label=game_id)

plt.xlabel("Time")
plt.ylabel("Score")
plt.title("Score over time per game")

plt.xticks(rotation=45)
plt.tight_layout()

# ⚠️ Only show legend if few games
# plt.legend()

plt.show()

<b>Solution 2</b>

In [ ]:
df1 = df.copy()

grouped_df = df1.groupby(df1.game_id)

'''
# Score progression over turns
for game_id, group in grouped_df:
    plt.plot(np.array(group['score'].to_list()))
'''

# Score progression over time
for game_id, group in grouped_df:
    plt.plot(group["timestamp"], group["score"], label=game_id)

plt.show()


## 2. Completion statistics

First filter the Dataframe to only include the ExitFoundEvent and GameEndedEvent. Next, count the records and show the plot.

<b>Solution 1</b>

In [ ]:
import matplotlib.pyplot as plt

filtered = df.loc[df['type'].isin(['ExitFoundEvent','GameEndedEvent'])]
counts = filtered["type"].value_counts()

counts.plot(kind="pie", autopct="%1d%%")
plt.title("Aantal per type")
plt.ylabel("")
plt.show()

<b>Solution 2</b>

In [ ]:
df2 = df.copy()

Series_type_count = df2["type"].value_counts()
spread = [Series_type_count["ExitFoundEvent"],Series_type_count["GameEndedEvent"]]
labels = ['finsihed', 'quit']

y = np.array(spread)

plt.pie(y, labels=labels, colors=["#69c5f6", "#feb240"])

## 3. Step counts per game

Show the amount of steps it took a player to reach the end of the game. Each game has a unique id. We are only interested in the games that actually found the exit.

In [ ]:
print(f"Total games: {(df["game_id"].nunique())}")
print(f"Total games with exit found: {df.loc[df["type"] == "ExitFoundEvent", "game_id"].count()}")

<b>Solution 1</b>

In [ ]:
games_with_exit = df.loc[df["type"] == "ExitFoundEvent", "game_id"].unique()
df_with_exit = df.loc[df["game_id"].isin(games_with_exit)]

df_with_exit = df_with_exit.loc[df["type"] != "GameStartedEvent"]

counts = df_with_exit["game_id"].value_counts()

counts.plot(kind="bar")
plt.title("Number of moves per game")
plt.xlabel("Game id")
plt.ylabel("# Moves")
plt.xticks(rotation=45)
plt.show()

<b>Solution 2</b>

In [ ]:
df3 = df.copy()

all_games = []
steps = 0

for i, row in df3.iterrows():
    steps += 1
    if row['message'] == 'You found the exit at specified height and width!' or row['message'] == 'Game ended by user.':
        all_games.append(steps)
        steps = 0

plt.hist(all_games, bins=30, color='#2ab0ff', edgecolor='#169acf', alpha=0.7)

## 4. Path visualization

Show the path of the game? A line over the grid, with a different color per game.

<b>Solution 1</b>

In [ ]:
import matplotlib.pyplot as plt

plt.figure()

for game_id, group in df.groupby("game_id"):
    group = group.sort_values("timestamp")
    
    plt.plot(
        group["column"],   # x-axis
        group["row"],      # y-axis
        marker="o",
        label=game_id
    )

plt.xlabel("Column")
plt.ylabel("Row")
plt.title("Game paths")

plt.gca().invert_yaxis()  # important for grid-like view
plt.show()

<b>Solution 2</b>

In [ ]:
df4 = df.copy()

grouped_df = df4.groupby(df4.game_id)

df_single_game = grouped_df.get_group('e0903690-74e7-4778-a1f5-b97e156eeecc')

x = np.array(df_single_game['row'])
y = np.array(df_single_game['column'])

plt.scatter(x,y)
plt.show()

## 5. Movement heatmap

Create a heatmap for the cells of the board. All records have a row,column, we know the size of the grid, create a heatmap for visits to a specific cell

<b>Solution 1</b>

In [ ]:
heatmap_data = (
    df.groupby(["row", "column"])
      .size()
      .unstack(fill_value=0)
)

import matplotlib.pyplot as plt

plt.imshow(heatmap_data, aspect='auto')

plt.colorbar(label="Visits")
plt.xlabel("Column")
plt.ylabel("Row")
plt.title("Heatmap of cell visits")

plt.show()

<b>Solution 2</b>

In [ ]:
df5 = df.copy()

df_board_heatmap = pd.DataFrame(
    0,
    columns=["0","1","2","3","4","5","6","7","8","9"],
    index=[0,1,2,3,4,5,6,7,8,9]
)

for i, row in df5.iterrows():
    df_board_heatmap.loc[row["row"], str(row["column"])] += 1


plt.imshow(df_board_heatmap, cmap='Purples', interpolation='nearest')

# Add colorbar
plt.colorbar()

plt.title("Heatmap of the gameboard")
plt.show()